# 01. 생활인구 EDA

## 분석 목적

부산 206개 행정동의 생활인구 특성을 분석하여
지역별 생활수요의 차이를 파악한다.

특히 다음을 확인한다.

1. 주거·직장·방문 생활인구의 지역별 분포
2. 행정동별 생활인구 구성 차이
3. 연령대별 생활수요의 공간적 차이
4. 공공서비스 공급 분석에 활용할 주요 수요 특성 선정

※ 분석기간: 2025.03 ~ 2025.12

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

folder = "/content/drive/MyDrive/부산_BigData_공모전/02_processed_data/"

print("===== 02_processed_data 파일 목록 =====")
for file in os.listdir(folder):
    print(file)

In [ ]:
# 전처리가 완료된 생활인구 데이터 불러오기

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

base_path = "/content/drive/MyDrive/부산_BigData_공모전/02_processed_data/"

pop_age = pd.read_csv(
    base_path + "생활인구_행정동연령별_202503_202512.csv"
)

pop_dong = pd.read_csv(
    base_path + "생활인구_행정동별_202503_202512.csv"
)

print("연령별 데이터:", pop_age.shape)
print("행정동별 데이터:", pop_dong.shape)

display(pop_dong.head())

In [ ]:
# 생활인구 변수의 전체 분포와 행정동 간 편차 확인

cols = [
    "평균주거인구수",
    "평균직장인구수",
    "평균방문인구수",
    "전체활동인구"
]

display(
    pop_dong[cols]
    .describe()
    .round(1)
    .T
)

In [ ]:
# 주거·직장·방문 생활인구 상위 행정동 비교

for col in ["평균주거인구수", "평균직장인구수", "평균방문인구수"]:
    print(f"\n===== {col} TOP 10 =====")

    display(
        pop_dong[
            ["행정동명", col]
        ]
        .sort_values(col, ascending=False)
        .head(10)
        .reset_index(drop=True)
    )

## 생활인구 기초분포 분석 결과

206개 행정동의 생활인구를 비교한 결과,
주거·직장·방문 생활인구 모두 행정동 간 편차가 크게 나타났다.

특히 평균이 중앙값보다 높고 최대값이 큰 모습을 보여,
일부 행정동에 생활인구가 집중되어 있음을 확인하였다.

생활인구 유형별 상위 지역도 서로 다르게 나타났다.

- 주거인구: 정관읍, 기장읍, 명지1동 등이 높음
- 직장인구: 녹산동, 우2동, 정관읍 등이 높음
- 방문인구: 부전2동, 대저2동, 우2동 등이 높음

따라서 단순한 전체 생활인구 규모만으로 지역수요를 판단하기보다
주거·직장·방문 생활인구의 구성 차이를 함께 고려할 필요가 있다.

다음 단계에서는 각 행정동의 생활인구 구성비를 계산하여
주거 중심·직장 중심·방문 중심 등의 지역 특성이 실제로 구분되는지 확인한다.

In [ ]:
# 전체 활동 규모에서 주거·직장·방문 생활인구가 차지하는 비율 계산

pop_dong["주거비율"] = (
    pop_dong["평균주거인구수"] / pop_dong["전체활동인구"]
)

pop_dong["직장비율"] = (
    pop_dong["평균직장인구수"] / pop_dong["전체활동인구"]
)

pop_dong["방문비율"] = (
    pop_dong["평균방문인구수"] / pop_dong["전체활동인구"]
)

display(
    pop_dong[
        ["행정동명", "전체활동인구", "주거비율", "직장비율", "방문비율"]
    ]
    .head()
    .round(3)
)

In [ ]:
# 생활인구 구성비가 높은 행정동 TOP 10 확인

for col in ["주거비율", "직장비율", "방문비율"]:
    print(f"\n===== {col} TOP 10 =====")

    display(
        pop_dong[
            ["행정동명", "전체활동인구", col]
        ]
        .sort_values(col, ascending=False)
        .head(10)
        .reset_index(drop=True)
        .round(3)
    )

## 생활인구 구성비 분석

행정동별 전체 활동 규모에서 주거·직장·방문 생활인구가 차지하는 비율을 비교하였다.

분석 결과 행정동에 따라 생활인구의 구성 특성이 뚜렷하게 다르게 나타났다.

- 주거비율이 높은 지역: 정관읍, 다대2동, 반여3동 등
- 직장비율이 높은 지역: 녹산동, 중앙동, 범일2동 등
- 방문비율이 높은 지역: 남포동, 대저2동, 부전2동 등

특히 전체활동인구 규모가 비슷하더라도 주거·직장·방문 구성은 서로 다를 수 있으므로,
지역수요 분석에서는 생활인구의 **규모와 구성 특성을 함께 고려**할 필요가 있다.

현재 단계에서는 임의의 기준값으로 지역을 유형화하지 않고
주거·직장·방문 비율을 연속형 특성으로 유지한다.

In [ ]:
# 연령대별 주거·직장·방문 생활인구의 평균 규모 확인

age_summary = (
    pop_age
    .groupby("나이대")[
        ["평균주거인구수", "평균직장인구수", "평균방문인구수"]
    ]
    .mean()
    .round(1)
)

display(age_summary)

In [ ]:
# 20대미만·60대이상 주거 생활인구를 행정동별로 추출

age_residential = (
    pop_age[
        pop_age["나이대"].isin(["20대미만", "60대이상"])
    ]
    .pivot(
        index=["행정동코드", "행정동명"],
        columns="나이대",
        values="평균주거인구수"
    )
    .reset_index()
)

age_residential.columns.name = None
age_residential = age_residential.rename(
    columns={
        "20대미만": "20대미만_주거인구",
        "60대이상": "60대이상_주거인구"
    }
)

display(age_residential.head())

In [ ]:
# 아동·청소년 및 60대 이상 주거수요 상위 행정동 확인

for col in ["20대미만_주거인구", "60대이상_주거인구"]:
    print(f"\n===== {col} TOP 10 =====")

    display(
        age_residential[
            ["행정동명", col]
        ]
        .sort_values(col, ascending=False)
        .head(10)
        .reset_index(drop=True)
        .round(1)
    )

## 연령대별 생활수요 기초분석

연령대별 생활인구를 비교한 결과,
60대이상 집단의 주거 및 방문 생활인구 규모가 크게 나타났다.

다만 `60대이상`은 다른 연령대와 달리 60세 이상의 여러 연령을 하나의 범주로 포함하므로,
다른 10년 단위 연령집단과 절대 규모를 직접 비교하여 고령수요가 가장 크다고 판단하지 않는다.

행정동별 주거 생활인구를 확인한 결과,

- 20대미만: 정관읍, 명지1동, 명지2동 등이 높음
- 60대이상: 기장읍, 정관읍, 금곡동, 다대1동 등이 높음

으로 나타났다.

특히 정관읍처럼 전체 주거인구 규모가 큰 지역은 여러 연령집단의 절대 인구수가 동시에 높게 나타날 수 있다.

따라서 특정 연령층의 생활수요를 판단할 때는
**절대 규모와 해당 연령층이 지역 주거인구에서 차지하는 비율을 함께 고려**할 필요가 있다.

In [ ]:
# 행정동별 연령대 주거인구와 연령 구성비 생성

age_wide = (
    pop_age
    .pivot(
        index=["행정동코드", "행정동명"],
        columns="나이대",
        values="평균주거인구수"
    )
    .reset_index()
)

age_wide.columns.name = None

age_cols = ["20대미만", "20대", "30대", "40대", "50대", "60대이상"]

age_wide["주거인구합계"] = age_wide[age_cols].sum(axis=1)

age_wide["20대미만_비율"] = (
    age_wide["20대미만"] / age_wide["주거인구합계"]
)

age_wide["60대이상_비율"] = (
    age_wide["60대이상"] / age_wide["주거인구합계"]
)

display(
    age_wide[
        ["행정동명", "주거인구합계",
         "20대미만", "20대미만_비율",
         "60대이상", "60대이상_비율"]
    ].head().round(3)
)

In [ ]:
# 20대미만·60대이상 수요의 절대 규모와 구성비 TOP 10 비교

for col in [
    "20대미만",
    "20대미만_비율",
    "60대이상",
    "60대이상_비율"
]:
    print(f"\n===== {col} TOP 10 =====")

    display(
        age_wide[
            ["행정동명", "주거인구합계", col]
        ]
        .sort_values(col, ascending=False)
        .head(10)
        .reset_index(drop=True)
        .round(3)
    )

## 연령수요의 절대 규모와 구성비 비교

특정 연령층의 생활수요를 보다 정확하게 파악하기 위해
행정동별 주거 생활인구의 절대 규모와 연령 구성비를 함께 비교하였다.

분석 결과 두 지표의 상위 지역이 서로 다르게 나타났다.

예를 들어 60대이상 주거인구의 절대 규모는
기장읍, 정관읍, 금곡동 등이 높게 나타난 반면,
60대이상 비율은 가락동, 철마면, 선두구동 등에서 높게 나타났다.

이는 다음과 같이 서로 다른 지역수요를 의미한다.

- 절대 규모가 높은 지역: 해당 연령층의 잠재적 서비스 이용자 자체가 많은 지역
- 구성비가 높은 지역: 지역 주민 구성에서 해당 연령층의 특성이 강한 지역

따라서 향후 공공서비스 수요 분석에서는
특정 연령층의 절대 규모와 구성비를 함께 고려한다.

단, 현재 생활인구 데이터의 `60대이상`은 60세 이상 전체를 하나의 범주로 제공하므로
이를 곧바로 법적·정책적 노인인구(예: 65세 이상)와 동일하게 해석하지 않는다.